<a href="https://colab.research.google.com/github/dororiya/zap-mission/blob/main/zap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# מערכת חכמה לזיהוי וסינון כפילויות מבוססת בינה מלאכותית (AI Deduplication Pipeline)

פרויקט זה משתמש בארכיטקטורה דו-שלבית המשלבת מודלים של שפה (LLMs) והטמעות וקטוריות (Embeddings) כדי לזהות, לקבץ ולסנן כפילויות של מוצרים, גם כאשר הם כתובים בשפות שונות או בתעתיק פונטי (למשל עברית ואנגלית).

---

## 🧠 איך המערכת עובדת?

### 1. למה BAAI/bge-m3? (הרשת הרחבה)
BAAI/bge-m3 הוא מודל הטמעות (Embedding) מתקדם שתוכנן במיוחד לטקסט רב-לשוני (Multilingual). "M3" מייצג רב-לשוניות (Multi-linguality), רב-גרנולריות (Multi-granularity), ורב-פונקציונליות (Multi-Functionality).
המודל לוקח מחרוזת טקסט (כמו "Apple MacBook Air M2") והופך אותה לווקטור מתמטי (רשימה של מספרים) במרחב רב-ממדי. אם שתי מחרוזות אומרות דברים דומים – אפילו בשפות שונות – הווקטורים שלהן יצביעו בערך לאותו כיוון.

* **למה אנחנו משתמשים בו:** הוא מהיר להפליא ורץ מקומית בחינם. הוא משמש כ"מעבר ראשון" כדי לפסול במהירות את ה-99% מהזוגות שלא דומים בכלל, ומעביר רק את הכפילויות הפוטנציאליות (אזור ה"אולי" המעורפל) לשלב הבא של ה-LLM.
* **כוח העל הרב-לשוני:** הוא מבין ש-"Apple" ו-"תפוח" קשורים מבחינה מתמטית, תכונה קריטית לסט הנתונים שלנו.

### 2. למה אלגוריתם Union-Find? (בונה הקבוצות / Buckets)
Union-Find (הידוע גם כ-Disjoint-Set) הוא מבנה נתונים קלאסי במדעי המחשב המשמש למעקב אחר איברים המחולקים לקבוצות נפרדות ללא חפיפה ("Buckets").

* **הבעיה:** אם איבר A דומה ל-B, ו-B דומה ל-C, אבל A ו-C נראים קצת שונים – בדיקה נאיבית של זוגות עלולה לחבר את A+B ואת B+C, אך להשאיר את A ו-C מופרדים.
* **הפתרון:** Union-Find פועל כמו תגובת שרשרת. אם "Apple MacBook" נוגע ב-"מאק בוק", ו-"מאק בוק" נוגע ב-"MacBook Air M2", האלגוריתם מקשר את כולם לקבוצה אחת גדולה. לאחר מכן, ה-LLM בוחן את כל הקבוצה בבת אחת, במקום להתבלבל מזוגות מקוטעים.

### 3. למה Groq ו-Llama 3.3 70B? (השופט הסופי)
הטמעות וקטוריות הן "חכמות" מבחינה מתמטית, אבל חסר להן היגיון בריא (Common Sense). מודל `bge-m3` עשוי לתת ציון דמיון גבוה ל-"iPhone 14" ול-"iPhone 14 Pro" כי המילים כמעט זהות, אבל בן אדם יודע שאלו מוצרים שונים לחלוטין עם פער מחירים של מאות שקלים.

* **למה אנחנו משתמשים בו:** אנחנו צריכים חשיבה דמוית-אדם כדי לקבל את ההחלטה הסופית. ל-Llama 3.3 70B (שרץ על החומרה המהירה-בטירוף של Groq) יש את היכולת הקוגניטיבית להבין חוקים נוקשים כמו "אל תקווץ דגמים שונים יחד", והוא יכול לגשר בבטחה על הפער שנוצר מתעתיקים פונטיים (כמו "סורפאס" ו-"Surface").

---

## 🚀 הוראות הרצה ודרישות מערכת (Run Instructions)

כיוון שהסקריפט משתמש במודלים כבדים, יש שתי דרכים מומלצות להריץ אותו: בסביבת ענן (מומלץ) או על המחשב המקומי. בשני המקרים, **תזדקקו למפתח API  של Groq**.

### אפשרות א': הרצה ב-Google Colab (מומלץ ביותר)
זוהי הדרך הטובה ביותר מכיוון שגוגל מספקת GPU וזיכרון RAM בחינם, מה שמונע קריסות של המחשב האישי.

1. פתחו מחברת חדשה ב-[Google Colab](https://colab.research.google.com/).
2. בתפריט העליון, לחצו על **Runtime (זמן ריצה)** -> **Change runtime type (שנה סוג זמן ריצה)**.
3. תחת **Hardware accelerator (מאיץ חומרה)**, בחרו ב-**T4 GPU** (זה יעניק לכם 16GB של זיכרון VRAM ייעודי להרצת ההטמעות במהירות) ולחצו על שמור.
4. הריצו את תא ההתקנות הבא:
   ```bash
   pip install sentence-transformers tqdm groq pandas
5. השיגו מפתח API מאתר Groq Console.
6. הזינו את מפתח ה-API שלכם בקוד, הריצו אותו.

**אפשרות ב': הרצה מקומית (Local Machine)**

אם אתם מריצים את הקוד על המחשב שלכם (למשל על Mac עם מעבד M-Series או PC רגיל), שימו לב לדרישות הבאות:

זיכרון RAM: נדרשים לפחות 8GB RAM פנויים (מומלץ 16GB) רק כדי לטעון ולהריץ את מודל bge-m3 בצורה חלקה בזיכרון המקומי.

GPU (אופציונלי אך מומלץ): * ב-Mac: הקוד מוגדר להשתמש ב-device='mps' כדי לנצל את המאיץ הגרפי של אפל.

ב-Windows/Linux: אם יש לכם כרטיס מסך של NVIDIA, ודאו ש-PyTorch מותקן עם תמיכה ב-CUDA (device='cuda'). אם אין, שנו את ההגדרה ל-device='cpu' (הריצה תהיה איטית יותר באופן משמעותי).

התקנות:

```bash
pip install sentence-transformers tqdm groq pandas

In [ ]:
# test list
products = [
    {"name": "Samsung Galaxy S23", "price": 799},
    {"name": "סמסונג גלקסי S23", "price": 749},
    {"name": "Samsung 23S Galaxy", "price": 819},
    {"name": "Apple MacBook Air M2", "price": 1199},
    {"name": "מאק בוק אייר M2", "price": 1149},
    {"name": "MacBook Air Apple M2", "price": 1229},
    {"name": "Sony WH-1000XM5", "price": 399},
    {"name": "סוני WH-1000XM5", "price": 379},
    {"name": "Sony 1000XM5 WH", "price": 409},
    {"name": "iPhone 14 Pro", "price": 999},
    {"name": "אייפון 14 פרו", "price": 969},
    {"name": "Apple iPhone 14 Pro", "price": 1019},
    {"name": "Dell XPS 15", "price": 1499},
    {"name": "דל XPS 15", "price": 1449},
    {"name": "XPS 15 Dell", "price": 1529},
    {"name": "Google Pixel 7", "price": 599},
    {"name": "גוגל פיקסל 7", "price": 569},
    {"name": "Pixel 7 Google", "price": 619},
    {"name": "Microsoft Surface Laptop 5", "price": 1099},
    {"name": "מיקרוסופט סורפאס 5", "price": 1049},
    {"name": "Surface Laptop 5 Microsoft", "price": 1129},
    {"name": "Bose QC45", "price": 329},
    {"name": "בוז QC45", "price": 309},
    {"name": "Bose QuietComfort 45", "price": 339},
    {"name": "LG C2 OLED TV", "price": 1299},
    {"name": "LG C2 OLED", "price": 1249},
    {"name": "LG OLED C2 55", "price": 1349},
    {"name": "Samsung QN90B", "price": 1499},
    {"name": "סמסונג QN90B", "price": 1449},
    {"name": "Samsung Neo QLED QN90B", "price": 1549},
    {"name": "iPad Pro 12.9", "price": 1099},
    {"name": "אייפד פרו 12.9", "price": 1049},
    {"name": "iPad Pro 2024", "price": 1129},
    {"name": "Galaxy Tab S8 Ultra", "price": 899},
    {"name": "גלקסי טאב S8 אולטרה", "price": 849},
    {"name": "Samsung Galaxy Tab S8 Ultra", "price": 929},
    {"name": "Kindle Paperwhite", "price": 149},
    {"name": "קינדל פייפרווייט", "price": 139},
    {"name": "Amazon Kindle Paperwhite", "price": 159},
    {"name": "Nintendo Switch OLED", "price": 349},
    {"name": "נינטנדו סוויץ' OLED", "price": 329},
    {"name": "Switch OLED Nintendo", "price": 359},
    {"name": "PS5", "price": 499},
    {"name": "פלייסטיישן 5", "price": 479},
    {"name": "PlayStation 5", "price": 519},
    {"name": "Xbox Series X", "price": 499},
    {"name": "אקס בוקס סיריס X", "price": 479},
    {"name": "Microsoft Xbox Series X", "price": 519},
    {"name": "AirPods Pro 2", "price": 249},
    {"name": "אייר פודס פרו 2", "price": 229},
    {"name": "Apple AirPods Pro 2", "price": 259},
    {"name": "Galaxy Buds 2 Pro", "price": 199},
    {"name": "גלקסי באדס 2 פרו", "price": 179},
    {"name": "Samsung Galaxy Buds 2 Pro", "price": 209},
    {"name": "Chromecast with Google TV", "price": 49},
    {"name": "כרומקאסט עם Google TV", "price": 44},
    {"name": "Google Chromecast 4K", "price": 54},
    {"name": "Roku Streaming Stick 4K", "price": 59},
    {"name": "רוקו סטרימינג סטיק 4K", "price": 54},
    {"name": "Roku 4K Stick", "price": 64},
    {"name": "Fire TV Stick 4K Max", "price": 55},
    {"name": "פייר טייוי סטיק 4K מקס", "price": 49},
    {"name": "Amazon Fire TV Stick 4K Max", "price": 59},
    {"name": "Echo Dot 5th gen", "price": 49},
    {"name": "אקו דוט דור 5", "price": 44},
    {"name": "Amazon Echo Dot 5", "price": 54},
    {"name": "Nest Audio", "price": 99},
    {"name": "נסט אודיו", "price": 89},
    {"name": "Google Nest Audio", "price": 109},
    {"name": "Fitbit Charge 5", "price": 149},
    {"name": "פיטבט צ'ארג' 5", "price": 139},
    {"name": "Fitbit Charge 5 black", "price": 159},
    {"name": "Garmin Venu 2", "price": 399},
    {"name": "גרמין וונו 2", "price": 379},
    {"name": "Garmin Venu 2 plus", "price": 419},
    {"name": "Ring Video Doorbell 4", "price": 199},
    {"name": "רינג וידאו דורבל 4", "price": 179},
    {"name": "Ring Doorbell 4", "price": 209},
    {"name": "Arlo Pro 4", "price": 199},
    {"name": "ארלו פרו 4", "price": 179},
    {"name": "Arlo Pro 4 camera", "price": 209},
    {"name": "TP-Link Kasa Smart Plug", "price": 19},
    {"name": "TP-Link Kasa פלאג חכם", "price": 17},
    {"name": "Kasa Smart Plug TP-Link", "price": 21},
    {"name": "Philips Hue White bulb", "price": 15},
    {"name": "פיליפס היואי נורה", "price": 13},
    {"name": "Philips Hue A19", "price": 17},
    {"name": "iRobot Roomba j7", "price": 599},
    {"name": "איירובוט רומבה j7", "price": 569},
    {"name": "Roomba j7 iRobot", "price": 629},
    {"name": "Dyson V15 Detect", "price": 699},
    {"name": "דייסון V15 דיטקט", "price": 659},
    {"name": "Dyson V15 vacuum", "price": 729},
    {"name": "Instant Pot Duo 6qt", "price": 89},
    {"name": "אינסטנט פוט דואו 6qt", "price": 79},
    {"name": "Instant Pot 6 quart", "price": 99},
    {"name": "Ninja Foodi Grill", "price": 199},
    {"name": "נינג'ה פודי גריל", "price": 179},
    {"name": "Ninja Grill Foodi", "price": 219},
    {"name": "Keurig K-Elite", "price": 149},
    {"name": "קוריג K-אליט", "price": 139},
    {"name": "Keurig Coffee Maker K-Elite", "price": 159},
    {"name": "Breville Barista Express", "price": 699},
    {"name": "ברוויל בריסטה אקספרס", "price": 659},
    {"name": "Breville Espresso Machine", "price": 729},
    {"name": "SodaStream Art", "price": 99},
    {"name": "סודהסטרים ארט", "price": 89},
    {"name": "SodaStream sparkling water maker", "price": 109},
    {"name": "Vitamix 5200", "price": 449},
    {"name": "ויטאמיקס 5200", "price": 419},
    {"name": "Vitamix blender 5200", "price": 479},
    {"name": "Shark IQ Robot Vacuum", "price": 399},
    {"name": "שארק IQ רובוט שואב", "price": 379},
    {"name": "Shark robot vacuum IQ", "price": 419},
    {"name": "Eufy RoboVac 11S", "price": 199},
    {"name": "יופי רובוואק 11S", "price": 179},
    {"name": "Eufy 11S RoboVac", "price": 219},
    {"name": "Anker PowerCore 26800", "price": 49},
    {"name": "אנקר פאוורקור 26800", "price": 44},
    {"name": "Anker portable charger 26800", "price": 54},
    {"name": "Belkin BoostCharge Pro", "price": 39},
    {"name": "בלקין בוסט צ'ארג' פרו", "price": 34},
    {"name": "Belkin wireless charger", "price": 44},
    {"name": "Samsung T7 SSD 1TB", "price": 119},
    {"name": "סמסונג T7 SSD 1TB", "price": 109},
    {"name": "Samsung external SSD T7", "price": 129},
    {"name": "SanDisk Extreme Pro USB", "price": 89},
    {"name": "סןדיסק אקסטרים פרו USB", "price": 79},
    {"name": "SanDisk 256GB Extreme Pro", "price": 99},
    {"name": "Seagate Expansion 2TB", "price": 79},
    {"name": "סיגייט אקספנשן 2TB", "price": 69},
    {"name": "Seagate external hard drive 2TB", "price": 89},
    {"name": "WD My Passport 1TB", "price": 59},
    {"name": "WD מיי פאספורט 1TB", "price": 54},
    {"name": "Western Digital My Passport", "price": 64},
    {"name": "Logitech MX Master 3", "price": 99},
    {"name": "לוג'יטק MX מאסטר 3", "price": 89},
    {"name": "Logitech mouse MX Master 3S", "price": 109},
    {"name": "Razer DeathAdder V2", "price": 69},
    {"name": "רייזר דת'אדר V2", "price": 59},
    {"name": "Razer gaming mouse DeathAdder", "price": 79},
    {"name": "Corsair K70 RGB", "price": 149},
    {"name": "קורסאייר K70 RGB", "price": 139},
    {"name": "Corsair mechanical keyboard K70", "price": 159},
    {"name": "SteelSeries Arctis 7", "price": 149},
    {"name": "סטיל סיריס ארקטיס 7", "price": 139},
    {"name": "SteelSeries wireless headset Arctis 7", "price": 159},
    {"name": "Blue Yeti USB mic", "price": 129},
    {"name": "בלו Yeti מיקרופון", "price": 119},
    {"name": "Blue Yeti microphone", "price": 139},
    {"name": "Elgato Stream Deck", "price": 149},
    {"name": "אלגטו סטרים דק", "price": 139},
    {"name": "Elgato Stream Deck MK2", "price": 159},
    {"name": "GoPro HERO11 Black", "price": 399},
    {"name": "גו פרו HERO11 בלאק", "price": 379},
    {"name": "GoPro Hero 11 Black", "price": 419},
    {"name": "DJI Mini 3 Pro", "price": 759},
    {"name": "DJI מיני 3 פרו", "price": 719},
    {"name": "DJI drone Mini 3 Pro", "price": 799},
    {"name": "Insta360 X3", "price": 449},
    {"name": "אינסטה 360 X3", "price": 419},
    {"name": "Insta360 360 camera X3", "price": 479},
    {"name": "Meta Quest 3", "price": 499},
    {"name": "מטא קווסט 3", "price": 479},
    {"name": "Oculus Quest 3", "price": 519},
    {"name": "Apple Watch Series 9", "price": 399},
    {"name": "אפל ווטש סיריס 9", "price": 379},
    {"name": "Apple Watch 9", "price": 419},
    {"name": "Samsung Galaxy Watch 6", "price": 299},
    {"name": "סמסונג גלקסי ווטש 6", "price": 279},
    {"name": "Galaxy Watch 6 Samsung", "price": 319},
    {"name": "Garmin Forerunner 265", "price": 449},
    {"name": "גרמין פורראנר 265", "price": 419},
    {"name": "Garmin running watch 265", "price": 479},
    {"name": "Whoop 4.0", "price": 239},
    {"name": "הופ 4.0", "price": 219},
    {"name": "Whoop fitness tracker", "price": 259},
    {"name": "Oura Ring Gen3", "price": 299},
    {"name": "אורה רינג Gen3", "price": 279},
    {"name": "Oura smart ring", "price": 319},
    {"name": "Ring Alarm Pro", "price": 249},
    {"name": "רינג אולם פרו", "price": 229},
    {"name": "Ring home security system", "price": 269},
    {"name": "SimpliSafe Home Security", "price": 199},
    {"name": "סימפלי סייף", "price": 179},
    {"name": "SimpliSafe alarm system", "price": 219},
    {"name": "Eero 6 mesh router", "price": 89},
    {"name": "אירו 6 רשת", "price": 79},
    {"name": "Amazon Eero 6", "price": 99},
    {"name": "TP-Link Deco X60", "price": 199},
    {"name": "TP-Link דקו X60", "price": 179},
    {"name": "Deco X60 mesh TP-Link", "price": 219},
    {"name": "Netgear Orbi RBK752", "price": 399},
    {"name": "נטגיר אורבי RBK752", "price": 379},
    {"name": "Orbi mesh Netgear", "price": 419},
    {"name": "Asus RT-AX86U", "price": 249},
    {"name": "אסוס RT-AX86U", "price": 229},
    {"name": "Asus gaming router AX86U", "price": 269},
    {"name": "Synology DS220+", "price": 299},
    {"name": "סינולוגיה DS220+", "price": 279},
    {"name": "Synology NAS 2 bay", "price": 319},
    {"name": "QNAP TS-253D", "price": 449},
    {"name": "קווינפ TS-253D", "price": 419},
    {"name": "QNAP NAS 2 bay", "price": 479},
    {"name": "Western Digital My Cloud", "price": 159},
    {"name": "WD מיי קלאוד", "price": 149},
    {"name": "My Cloud home NAS", "price": 169},
    {"name": "Raspberry Pi 4 4GB", "price": 55},
    {"name": "ראספברי פ�י 4 4GB", "price": 49},
    {"name": "Raspberry Pi 4 model B", "price": 59},
    {"name": "Arduino Uno R3", "price": 27},
    {"name": "ארדואינו אונו R3", "price": 24},
    {"name": "Arduino Uno board", "price": 29},
    {"name": "ESP32 dev board", "price": 12},
    {"name": "ESP32 לוח פיתוח", "price": 10},
    {"name": "ESP32 WiFi Bluetooth", "price": 14},
    {"name": "Canon EOS R10", "price": 999},
    {"name": "קנון EOS R10", "price": 949},
    {"name": "Canon mirrorless R10", "price": 1049},
    {"name": "Nikon Z50", "price": 849},
    {"name": "ניקון Z50", "price": 799},
    {"name": "Nikon mirrorless Z50", "price": 899},
    {"name": "Sony Alpha a6400", "price": 899},
    {"name": "סוני אלפא a6400", "price": 849},
    {"name": "Sony mirrorless a6400", "price": 949},
    {"name": "Fujifilm X-T5", "price": 1699},
    {"name": "פוג'יפילם X-T5", "price": 1649},
    {"name": "Fujifilm mirrorless XT5", "price": 1749},
    {"name": "Panasonic Lumix GH6", "price": 1899},
    {"name": "פנסוניק לומיקס GH6", "price": 1849},
    {"name": "Lumix GH6 Panasonic", "price": 1949},
    {"name": "DJI Osmo Mobile 6", "price": 159},
    {"name": "DJI אוסמו מובייל 6", "price": 149},
    {"name": "Osmo Mobile gimbal DJI", "price": 169},
    {"name": "Zhiyun Smooth 5", "price": 139},
    {"name": "ז'ייון סמות' 5", "price": 129},
    {"name": "Zhiyun smartphone gimbal", "price": 149},
    {"name": "Rode VideoMic GO II", "price": 99},
    {"name": "רוד וידאומיק GO II", "price": 89},
    {"name": "Rode shotgun mic", "price": 109},
    {"name": "Shure MV7", "price": 249},
    {"name": "שור MV7", "price": 229},
    {"name": "Shure podcast mic", "price": 269},
    {"name": "Elgato Key Light Air", "price": 99},
    {"name": "אלגטו קיי לייט אייר", "price": 89},
    {"name": "Elgato streaming light", "price": 109},
    {"name": "Nanoleaf Shapes", "price": 199},
    {"name": "ננוליף שייפס", "price": 179},
    {"name": "Nanoleaf smart lights", "price": 219},
    {"name": "LIFX Color A60", "price": 49},
    {"name": "ליפקס קלור A60", "price": 44},
    {"name": "LIFX smart bulb", "price": 54},
    {"name": "Govee RGBIC strip", "price": 39},
    {"name": "גובי RGBIC סטריפ", "price": 34},
    {"name": "Govee LED strip lights", "price": 44},
    {"name": "Wyze Cam v3", "price": 35},
    {"name": "וייז קאם v3", "price": 29},
    {"name": "Wyze indoor camera", "price": 39},
    {"name": "Blink Outdoor 4", "price": 99},
    {"name": "בלינק אאוטדור 4", "price": 89},
    {"name": "Blink security camera", "price": 109},
    {"name": "Eufy SoloCam S40", "price": 199},
    {"name": "יופי סולוקאם S40", "price": 179},
    {"name": "Eufy solar camera", "price": 219},
    {"name": "Reolink Argus 3 Pro", "price": 109},
    {"name": "ריאולינק ארגוס 3 פרו", "price": 99},
    {"name": "Reolink wireless camera", "price": 119},
    {"name": "Nest Thermostat E", "price": 169},
    {"name": "נסט תרמוסטט E", "price": 159},
    {"name": "Google Nest Thermostat", "price": 179},
    {"name": "Ecobee Smart Thermostat", "price": 219},
    {"name": "אקובי סמארט תרמוסטט", "price": 199},
    {"name": "Ecobee with sensor", "price": 239},
    {"name": "Honeywell Home T9", "price": 199},
    {"name": "האניוול הום T9", "price": 179},
    {"name": "Honeywell smart thermostat", "price": 219},
    {"name": "August Wi-Fi Smart Lock", "price": 199},
    {"name": "אוגוסט Wi-Fi סמארט לוק", "price": 179},
    {"name": "August smart lock 4th gen", "price": 219},
    {"name": "Yale Assure Lock 2", "price": 159},
    {"name": "ייל אשור לוק 2", "price": 149},
    {"name": "Yale smart lock", "price": 169},
    {"name": "Schlage Encode Plus", "price": 299},
    {"name": "שלג' אנקוד פלוס", "price": 279},
    {"name": "Schlage WiFi deadbolt", "price": 319},
    {"name": "MyQ Smart Garage Hub", "price": 29},
    {"name": "מיי קיו סמארט גראז'", "price": 24},
    {"name": "Chamberlain MyQ", "price": 34},
    {"name": "Tailwind iQ3", "price": 79},
    {"name": "טיילוינד iQ3", "price": 69},
    {"name": "Tailwind smart garage controller", "price": 89},
    {"name": "Aqara Hub M2", "price": 59},
    {"name": "אקוורה האב M2", "price": 54},
    {"name": "Aqara smart home hub", "price": 64},
    {"name": "SwitchBot Hub 2", "price": 49},
    {"name": "סוויצ'בוט האב 2", "price": 44},
    {"name": "SwitchBot IR blaster", "price": 54},
    {"name": "Bond Bridge", "price": 99},
    {"name": "בונד ברידג'", "price": 89},
    {"name": "Bond smart ceiling fan controller", "price": 109},
    {"name": "Broadlink RM4 Pro", "price": 45},
    {"name": "ברודלינק RM4 פרו", "price": 39},
    {"name": "Broadlink universal remote", "price": 49},
    {"name": "Sonos One SL", "price": 199},
    {"name": "סונוס וואן SL", "price": 179},
    {"name": "Sonos speaker", "price": 219},
    {"name": "Bose SoundLink Revolve+", "price": 299},
    {"name": "בוז סאונדלינק ריבולב+", "price": 279},
    {"name": "Bose portable speaker", "price": 319},
    {"name": "JBL Charge 5", "price": 179},
    {"name": "JBL צ'ארג' 5", "price": 159},
    {"name": "JBL waterproof speaker", "price": 199},
    {"name": "Ultimate Ears Boom 3", "price": 149},
    {"name": "אולטימייט אירס בום 3", "price": 139},
    {"name": "UE Boom 3", "price": 159},
    {"name": "Marshall Stanmore II", "price": 349},
    {"name": "מרשל סטנמור II", "price": 329},
    {"name": "Marshall Bluetooth speaker", "price": 369},
    {"name": "Harman Kardon Onyx Studio 7", "price": 249},
    {"name": "הרמן קרדון אוניקס סטודיו 7", "price": 229},
    {"name": "Harman Kardon wireless speaker", "price": 269},
    {"name": "Anker Soundcore Motion+", "price": 99},
    {"name": "אנקר סאונדקור מושן+", "price": 89},
    {"name": "Soundcore Motion plus", "price": 109},
    {"name": "Tribit StormBox Blast", "price": 159},
    {"name": "טריביט סטורםבוקס בלאסט", "price": 149},
    {"name": "Tribit speaker", "price": 169},
    {"name": "Bang & Olufsen Beosound A1", "price": 249},
    {"name": "באונג & אולופסן ביאוסאונד A1", "price": 229},
    {"name": "B&O portable speaker", "price": 269},
    {"name": "Sony SRS-XB43", "price": 199},
    {"name": "סוני SRS-XB43", "price": 179},
    {"name": "Sony extra bass speaker", "price": 219},
    {"name": "LG XBoom RN9", "price": 299},
    {"name": "LG XBoom RN9", "price": 279},
    {"name": "LG party speaker", "price": 319},
    {"name": "Vizio M-Series 5.1", "price": 349},
    {"name": "ויזיו M-סיריס 5.1", "price": 329},
    {"name": "Vizio soundbar", "price": 369},
    {"name": "Samsung HW-Q990B", "price": 1199},
    {"name": "סמסונג HW-Q990B", "price": 1149},
    {"name": "Samsung soundbar", "price": 1249},
    {"name": "Sonos Beam Gen 2", "price": 449},
    {"name": "סונוס בים ג'ן 2", "price": 429},
    {"name": "Sonos soundbar", "price": 479},
    {"name": "Bose Smart Soundbar 600", "price": 499},
    {"name": "בוז סמארט סאונדבר 600", "price": 479},
    {"name": "Bose soundbar", "price": 529},
    {"name": "Apple TV 4K (2022)", "price": 129},
    {"name": "אפל TV 4K 2022", "price": 119},
    {"name": "Apple TV 4K latest", "price": 139},
    {"name": "Nvidia Shield TV Pro", "price": 199},
    {"name": "אנבידיה שילד TV פרו", "price": 179},
    {"name": "Nvidia Shield streaming", "price": 219},
    {"name": "Walmart Onn 4K box", "price": 19},
    {"name": "וולמארט און 4K", "price": 17},
    {"name": "Onn Google TV 4K", "price": 21},
    {"name": "TiVo Stream 4K", "price": 39},
    {"name": "טיוו סטרים 4K", "price": 34},
    {"name": "TiVo streaming stick", "price": 44}
]

In [ ]:
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
import json
import time
from groq import Groq
from tqdm import tqdm

# --- CONFIGURATION ---
# Paste your Groq API Key here
GROQ_API_KEY = ""

# Initialize the Groq Client
client = Groq(api_key=GROQ_API_KEY)

# --- STAGE 1: LOCAL EMBEDDINGS ON COLAB GPU ---
print("Loading BGE-M3 model onto Colab GPU...")
model = SentenceTransformer('BAAI/bge-m3')
names = [p["name"] for p in products]

print("\nGenerating embeddings...")
embeddings = model.encode(names, convert_to_tensor=True, show_progress_bar=True)

# Union-Find for Bucketing
parent = list(range(len(names)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent[find(x)] = find(y)

BUCKET_THRESHOLD = 0.55

total_pairs = (len(names) * (len(names) - 1)) // 2
with tqdm(total=total_pairs, desc="Grouping items") as pbar:
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            sim = util.cos_sim(embeddings[i], embeddings[j]).item()
            if sim > BUCKET_THRESHOLD:
                union(i, j)
            pbar.update(1)

buckets = defaultdict(list)
for idx in range(len(names)):
    buckets[find(idx)].append(products[idx])

final_unique_products = []

# --- STAGE 2: GROQ API VERIFICATION ---
print(f"\nCreated {len(buckets)} candidate groups. Sending to Groq API...")

for root, bucket in tqdm(buckets.items(), desc="LLM Verifying Groups"):
    if len(bucket) == 1:
        final_unique_products.append(bucket[0])
        continue

    prompt = f"""You are an e-commerce data cleaner.
Your rules:
1. Group EXACT duplicates together (including Hebrew/English phonetics like 'סורפאס' and 'Surface').
2. DO NOT group different models together (e.g., 'iPhone 14' is different from 'iPhone 14 Pro').
3. For each group of exact duplicates, find the LOWEST price.
4. You MUST output your final answer as a raw JSON array of objects.
5. Do not include markdown formatting or explanations.

Example Output Format:
[
  {{"name": "Clean English Name", "price": 100}}
]

Data:
{json.dumps(bucket, ensure_ascii=False)}
"""

    try:
        # We use Llama-3.3-70b - a massive, highly capable model running instantly on Groq
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.3-70b-versatile",
            temperature=0,
            # Force Groq to return strict JSON
            response_format={"type": "json_object"}
        )

        # Groq's JSON mode returns an object, so we wrap the array instruction slightly
        # We expect {"products": [...] }
        result = chat_completion.choices[0].message.content
        cleaned_bucket = json.loads(result)

        # Depending on how the LLM formatted it, handle arrays or nested objects
        if isinstance(cleaned_bucket, list):
            final_unique_products.extend(cleaned_bucket)
        elif isinstance(cleaned_bucket, dict):
            # Try to find the array inside the dict
            for key, val in cleaned_bucket.items():
                if isinstance(val, list):
                    final_unique_products.extend(val)
                    break
            else:
                final_unique_products.append(cleaned_bucket)

        # Groq's free tier has a rate limit (usually 30 requests per minute).
        # A tiny 2-second sleep prevents hitting the limit when processing many buckets.
        time.sleep(2)

    except Exception as e:
        print(f"\nAPI/Parse Error on bucket: {e}")
        final_unique_products.extend(bucket)

print(f"\nFinal count: {len(final_unique_products)} unique products out of {len(products)} original items.")
print(json.dumps(final_unique_products, indent=2, ensure_ascii=False))

Loading BGE-M3 model onto Colab GPU...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Generating embeddings...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Grouping items: 100%|██████████| 64620/64620 [00:14<00:00, 4548.29it/s]



Created 16 candidate groups. Sending to Groq API...


LLM Verifying Groups: 100%|██████████| 16/16 [00:35<00:00,  2.24s/it]


Final count: 128 unique products out of 360 original items.
[
  {
    "name": "Samsung Galaxy S23",
    "price": 749
  },
  {
    "name": "Apple MacBook Air M2",
    "price": 1149
  },
  {
    "name": "Sony WH-1000XM5",
    "price": 379
  },
  {
    "name": "iPhone 14 Pro",
    "price": 969
  },
  {
    "name": "Dell XPS 15",
    "price": 1449
  },
  {
    "name": "Google Pixel 7",
    "price": 569
  },
  {
    "name": "Microsoft Surface Laptop 5",
    "price": 1049
  },
  {
    "name": "Bose QC45",
    "price": 309
  },
  {
    "name": "LG C2 OLED TV",
    "price": 1249
  },
  {
    "name": "Samsung QN90B",
    "price": 1449
  },
  {
    "name": "iPad Pro 12.9",
    "price": 1049
  },
  {
    "name": "Nintendo Switch OLED",
    "price": 329
  },
  {
    "name": "PS5",
    "price": 479
  },
  {
    "name": "Xbox Series X",
    "price": 479
  },
  {
    "name": "AirPods Pro 2",
    "price": 229
  },
  {
    "name": "Galaxy Buds 2 Pro",
    "price": 179
  },
  {
    "name": "Chromecast 